# 🎭 Telugu Poetry Generation - Google Colab Training
**Project**: CNN-Based Poem Learning & Interpretation Inspired by Human Rote Learning  
**Language**: Telugu (తెలుగు)  
**GPU**: Google Colab T4/V100 (Free/Pro)  
**Estimated Time**: 2-3 hours

---

## 📌 How This Works

✅ **Takes Project FROM**: GitHub (fully automated clone)  
✅ **Saves Results TO**: Google Drive (for download after training)  
✅ **Runs ON**: Colab GPU (free!)  

**No local files needed - everything is pulled from GitHub!**

---

## ✅ What This Notebook Does

1. ✅ Clone GitHub repository
2. ✅ Install all dependencies
3. ✅ Download Telugu poem datasets (9,000+ poems)
4. ✅ Train enhanced model on GPU
5. ✅ Evaluate generation quality
6. ✅ Save checkpoints to Google Drive

---

## 📌 Prerequisites

- Google Colab account (free)
- Google Drive (optional - for saving results)
- 2-3 hours GPU time

**Estimated GPU Hours Used**: ~3 hours (T4) or ~1.5 hours (V100)

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: GPU not available. Change runtime to GPU!")
    print("Menu: Runtime → Change runtime type → Hardware accelerator: GPU")

In [ ]:
# Mount Google Drive (OPTIONAL - only if you want to save results to Drive)
# If you just want to train and download from Colab, you can skip this

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
    drive_available = True
except Exception as e:
    print(f"⚠️  Google Drive not mounted: {str(e)}")
    print("   This is OK - results can still be downloaded directly from Colab")
    drive_available = False

## 📦 Step 2: Clone Repository

In [ ]:
import os
import subprocess
from pathlib import Path

# Clone the GitHub repository
repo_url = "https://github.com/maneendra03/CNN-Based-Telugu-Poem-Analysis-inspired-by-human-rote-learning.git"
repo_name = "telugu-poetry-ai"

if not os.path.exists(repo_name):
    print(f"🔄 Cloning repository...")
    subprocess.run(["git", "clone", repo_url, repo_name], check=True)
    print(f"✅ Repository cloned successfully!")
else:
    print(f"✅ Repository already exists")

# Change to project directory
os.chdir(repo_name)
print(f"📁 Current directory: {os.getcwd()}")
print(f"📂 Files: {os.listdir('.')[:10]}...")

## 📥 Step 3: Install Dependencies

In [ ]:
# Install requirements
print("📦 Installing dependencies...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ All dependencies installed!")

# Install additional Colab-specific packages
print("\n📦 Installing additional packages...")
subprocess.run(["pip", "install", "-q", "kaggle", "huggingface-hub"], check=True)
print("✅ Additional packages installed!")

## 📊 Step 4: Download Datasets

In [ ]:
# Download datasets from HuggingFace and Kaggle
print("📥 Downloading Telugu poem datasets...\n")

# Download from HuggingFace (5,115 poems)
print("1️⃣  Downloading from HuggingFace (SuryaKrishna02/aya-telugu-poems)...")
result = subprocess.run(
    ["python", "scripts/download_datasets.py", "--skip-kaggle"],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"⚠️  Warning: {result.stderr}")

print("\n✅ Dataset download complete!")
print(f"\n📂 Checking processed data...")
import json
if os.path.exists('data/processed/telugu_train.json'):
    with open('data/processed/telugu_train.json') as f:
        train_data = json.load(f)
    print(f"   ✅ Train set: {len(train_data.get('poems', train_data)) if isinstance(train_data, dict) else len(train_data)} poems")
if os.path.exists('data/processed/telugu_val.json'):
    with open('data/processed/telugu_val.json') as f:
        val_data = json.load(f)
    print(f"   ✅ Val set: {len(val_data.get('poems', val_data)) if isinstance(val_data, dict) else len(val_data)} poems")

## 🚀 Step 5: Train the Model

This will train the enhanced generator (V3) with:
- Coverage attention (prevents repetition)
- Style conditioning (meter, rasa, theme)
- Anti-repetition mechanisms
- Multiple loss functions

In [ ]:
import sys
sys.path.insert(0, '/content/telugu-poetry-ai')

# Import training components
import yaml
import torch
from pathlib import Path

# Load config
with open('config/config.yaml') as f:
    config = yaml.safe_load(f)

print("⚙️  Training Configuration:")
print(f"  - Learning Rate: {config.get('learning_rate', 0.001)}")
print(f"  - Batch Size: {config.get('batch_size', 8)}")
print(f"  - Epochs: {config.get('epochs', 10)}")
print(f"  - Model Type: Enhanced Generator V3")
print(f"  - Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print("\n🚀 Starting training...\n")

In [ ]:
# Run training script
print("Training with telugu_training_v3 configuration...\n")

# Execute training
result = subprocess.run(
    ["python", "scripts/train_telugu_v3.py", 
     "--epochs", "15",
     "--batch-size", "8",
     "--learning-rate", "0.001",
     "--device", "cuda"],
    capture_output=True,
    text=True,
    timeout=10800  # 3 hours timeout
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print("\n✅ Training completed successfully!")
else:
    print(f"\n⚠️  Training finished with code {result.returncode}")

## 📈 Step 6: Evaluate Results

In [ ]:
import json
from pathlib import Path

print("📊 EVALUATION RESULTS\n" + "="*60)

# Load evaluation results
results_path = Path('results/evaluation_results.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    
    print("\n🎯 Main Metrics:")
    for key, value in results.items():
        if isinstance(value, float):
            print(f"   {key}: {value:.4f}")
        else:
            print(f"   {key}: {value}")
else:
    print("⚠️  evaluation_results.json not found yet")

# Check checkpoint
checkpoint_dir = Path('checkpoints/telugu')
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob('*.pt'))
    if checkpoints:
        print(f"\n💾 Saved Checkpoints:")
        for ckpt in sorted(checkpoints)[-3:]:
            size = ckpt.stat().st_size / (1024**2)
            print(f"   - {ckpt.name} ({size:.1f} MB)")

## 🎨 Step 7: Test Generation

In [ ]:
# Test poem generation
print("🎭 Testing Poem Generation\n" + "="*60)

try:
    from src.models.enhanced_generator import EnhancedGenerator
    from src.preprocessing.telugu_cleaner import TeluguCleaner
    
    # Initialize generator
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    generator = EnhancedGenerator(device=device)
    
    # Load checkpoint if exists
    checkpoint_dir = Path('checkpoints/telugu')
    if checkpoint_dir.exists():
        checkpoints = list(checkpoint_dir.glob('*.pt'))
        if checkpoints:
            latest_ckpt = sorted(checkpoints)[-1]
            print(f"📂 Loading checkpoint: {latest_ckpt.name}\n")
            generator.load(str(latest_ckpt))
    
    # Test generation
    test_prompts = [
        "చందమామ",  # Moon
        "ప్రేమ",     # Love
        "కవితా",    # Poetry
    ]
    
    for prompt in test_prompts:
        print(f"📝 Prompt: {prompt}")
        try:
            poem = generator.generate(prompt, max_length=100)
            print(f"   Generated:\n   {poem}\n")
        except Exception as e:
            print(f"   ⚠️  Generation error: {str(e)[:100]}...\n")
            
except Exception as e:
    print(f"⚠️  Could not test generation: {str(e)[:200]}")
    print("   This is normal if model files are not fully saved yet.")

## 💾 Step 8: Save Results to Google Drive

In [ ]:
import shutil
from pathlib import Path

print("💾 Saving Results...\n")

# Check if Google Drive is available
if drive_available:
    print("📂 Saving to Google Drive...")
    
    # Create results directory in Google Drive
    drive_results = Path('/content/drive/MyDrive/telugu-poetry-training-results')
    drive_results.mkdir(parents=True, exist_ok=True)
    
    # Copy key files
    files_to_copy = [
        ('results/evaluation_results.json', 'evaluation_results.json'),
        ('results/validation_results.json', 'validation_results.json'),
        ('config/config.yaml', 'config.yaml'),
    ]
    
    print("📂 Copied to Google Drive:")
    for src, dst in files_to_copy:
        src_path = Path(src)
        if src_path.exists():
            dst_path = drive_results / dst
            shutil.copy(src_path, dst_path)
            print(f"   ✅ {dst}")
        else:
            print(f"   ⏭️  {src} (not found)")
    
    # Copy latest checkpoint
    checkpoint_dir = Path('checkpoints/telugu')
    if checkpoint_dir.exists():
        checkpoints = list(checkpoint_dir.glob('*.pt'))
        if checkpoints:
            latest_ckpt = sorted(checkpoints)[-1]
            dst_path = drive_results / f"checkpoint_{latest_ckpt.name}"
            shutil.copy(latest_ckpt, dst_path)
            print(f"   ✅ {latest_ckpt.name} ({latest_ckpt.stat().st_size / (1024**2):.1f} MB)")
    
    print(f"\n📁 Results saved to: /content/drive/MyDrive/telugu-poetry-training-results")
    print("✅ Download from Google Drive!")
else:
    print("📂 Google Drive not available - Files stay in Colab storage")
    print("   You can still download them using Colab's download feature:")
    print("   Files → Download")
    
print("\n✅ Results ready for download!")

## 📊 Step 9: Create Performance Report

In [ ]:
import json
from pathlib import Path
from datetime import datetime

# Create summary report
report = {
    "training_date": datetime.now().isoformat(),
    "environment": {
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
    },
    "model_config": {
        "type": "Enhanced Generator V3",
        "features": [
            "Coverage Attention",
            "Style Conditioning",
            "N-gram Blocking",
            "Anti-repetition Loss"
        ]
    }
}

# Load actual results if available
if Path('results/evaluation_results.json').exists():
    with open('results/evaluation_results.json') as f:
        report['evaluation_results'] = json.load(f)

# Save report
report_path = Path('results/training_report.json')
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("📊 Training Report Generated:")
print(json.dumps(report, indent=2, ensure_ascii=False))

print(f"\n✅ Report saved to: results/training_report.json")

## ✅ Training Complete!

### 📊 Summary

✅ **Completed Steps:**
1. ✅ Cloned repository from GitHub
2. ✅ Installed all dependencies
3. ✅ Downloaded 9,000+ Telugu poems
4. ✅ Trained Enhanced Generator V3
5. ✅ Evaluated generation quality
6. ✅ Saved checkpoints and results

### 📈 Next Steps

1. **Download Results**: Go to Google Drive → `telugu-poetry-training-results/`
2. **Check Metrics**: Open `evaluation_results.json`
3. **Download Checkpoint**: Save `checkpoint_*.pt` to your local machine
4. **Use Locally**: Copy checkpoint and run:
   ```bash
   python app/telugu_ui.py --checkpoint checkpoints/telugu/model_latest.pt
   ```

### 🔗 Project Links

- **GitHub**: https://github.com/maneendra03/CNN-Based-Telugu-Poem-Analysis-inspired-by-human-rote-learning
- **Local**: `/Users/mani/Desktop/majorproject - A`

---

**Notebook created**: January 31, 2026  
**Estimated GPU time**: 2-3 hours (T4) | 1-1.5 hours (V100)

💡 **Tip**: You can run this notebook multiple times to train on different epochs or configurations!